# NIFTY 50 + Financial News — End-to-End Research and Forecast Audit

**Purpose:** a detailed, reproducible audit notebook for the `finance-IDC-Project`. It documents the data inventory, full-corpus versus monthly-capped news, feature construction, model training, expanding versus rolling walk-forward validation, PCA, the adaptive selector, and the limits of the current forecast.

This notebook is **analysis-only**: it does not fetch prices, scrape news, run FinBERT, rebuild datasets, alter the adaptive configuration, or write to the forecast database. It reads files produced by the pipeline and displays/analyses them. If an output file is absent, the relevant section reports that rather than inventing a result.

**Official modelling choice:** `model_dataset_sampled_monthly_2026.csv`, with a maximum of 500 articles per calendar month in 2026. The full-corpus dataset remains useful as a sensitivity/research comparison, not the official forecasting input.

**Important:** historical walk-forward performance has not shown a feature model reliably beating the mean-return baseline. This is a research prototype, not evidence of a profitable trading strategy.

## Notebook roadmap

The notebook follows the project workflow from raw data checks to forecast interpretation:

1. **Setup and project files** — locate inputs and outputs, load datasets and reports.
2. **Data quality and sampling** — inspect coverage, missingness, duplicates, and the 2026 monthly article cap.
3. **Feature exploration and correlations** — inspect distributions, then compare Pearson and Spearman associations.
4. **Forecast targets and validation** — define regression/classification targets and review out-of-sample walk-forward results.
5. **Walk-forward diagnostics** — compare expanding/rolling windows, period metrics, predictions, and residuals.
6. **PCA evaluation** — inspect variance/loadings and use a leakage-safe walk-forward template; PCA is not adopted without evidence of improvement.
7. **Adaptive selection and official forecast** — review selector decisions, latest forecast, and prospective outcome tracking.
8. **Leakage, conclusions, and audit checklist** — verify cutoff timing, limitations, and reproducibility.

**Important:** run the notebook from top to bottom after restarting the kernel. Sections that depend on optional report files will explain when a file is unavailable rather than inventing results.


## How to run this notebook

1. Save this notebook in the project root or in a subfolder of the project.
2. Open Jupyter using the `finance-IDC` environment.
3. Run **Kernel → Restart Kernel and Run All Cells**.
4. If the project is not found automatically, set `PROJECT_ROOT` in the next cell.

Dependencies are the usual project analysis stack: `pandas`, `numpy`, `matplotlib`, and `scikit-learn`. Most sections work without scikit-learn; the PCA section checks for it.

In [ ]:
from pathlib import Path
import json, warnings, textwrap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 180)
pd.set_option("display.max_colwidth", 120)

KNOWN_ROOT = Path(r"C:\Users\surya\Documents\finance-IDC-Project")
if (Path.cwd() / "data" / "processed").exists():
    PROJECT_ROOT = Path.cwd()
elif (KNOWN_ROOT / "data" / "processed").exists():
    PROJECT_ROOT = KNOWN_ROOT
else:
    PROJECT_ROOT = Path.cwd()  # Edit this if the project is elsewhere.

DATA = PROJECT_ROOT / "data" / "processed"
print("Notebook cwd :", Path.cwd())
print("Project root :", PROJECT_ROOT)
print("Processed dir:", DATA, "| exists:", DATA.exists())
if not DATA.exists():
    raise FileNotFoundError("Set PROJECT_ROOT to the finance-IDC-Project directory.")

## Project map and files used

The notebook deliberately separates **inputs**, **experiment outputs**, **official forecast outputs**, and **prospective outcome tracking**. The pipeline is responsible for generating/updating these files; this notebook reads them.

In [ ]:
FILES = {
    "full_dataset": "model_dataset.csv",
    "capped_dataset": "model_dataset_sampled_monthly_2026.csv",
    "news_scored": "news_sentiment_scored.csv",
    "news_capped": "news_sentiment_sampled_monthly_2026.csv",
    "daily_news_full": "news_sentiment_daily.csv",
    "daily_news_capped": "news_sentiment_daily_sampled_monthly_2026.csv",
    "market_daily": "news_market_daily.csv",
    "return_comparison": "monthly_sampling_return_comparison.csv",
    "direction_comparison": "monthly_sampling_direction_comparison.csv",
    "paired_predictions": "monthly_sampling_paired_predictions.csv",
    "period_metrics": "monthly_sampling_period_metrics.csv",
    "window_comparison": "walk_forward_window_comparison.csv",
    "walk_predictions": "walk_forward_predictions.csv",
    "adaptive_metrics": "adaptive_model_selection.csv",
    "adaptive_config": "adaptive_model_selection.json",
    "forecast": "latest_forecast.csv",
    "forecast_comparison": "latest_forecast_comparison.csv",
    "outcome_history": "forecast_outcome_history.csv",
    "price_history": "market_prices.csv",
}

def path_for(key):
    return DATA / FILES[key]

inventory = []
for key, filename in FILES.items():
    p = DATA / filename
    inventory.append({"key": key, "file": filename, "exists": p.exists(),
                      "size_kb": round(p.stat().st_size/1024, 1) if p.exists() else np.nan})
display(pd.DataFrame(inventory))

## Load datasets and experiment reports

The model datasets contain one row per market session, not one row per news article. The news tables contain article-level or daily-aggregated records. Do not compare their row counts as though they represent the same unit of observation.

In [ ]:
def read_csv_if_exists(key, parse_dates=None):
    p = path_for(key)
    if not p.exists():
        print(f"Missing optional file: {p.name}")
        return None
    try:
        return pd.read_csv(p, parse_dates=parse_dates, low_memory=False)
    except Exception as exc:
        print(f"Could not read {p.name}: {exc}")
        return None

def read_json_if_exists(key):
    p = path_for(key)
    if not p.exists():
        print(f"Missing optional file: {p.name}")
        return None
    try:
        return json.loads(p.read_text(encoding="utf-8"))
    except Exception as exc:
        print(f"Could not read {p.name}: {exc}")
        return None

full_df = read_csv_if_exists("full_dataset")
capped_df = read_csv_if_exists("capped_dataset")
scored_news = read_csv_if_exists("news_scored")
capped_news = read_csv_if_exists("news_capped")
daily_full = read_csv_if_exists("daily_news_full")
daily_capped = read_csv_if_exists("daily_news_capped")
market_daily = read_csv_if_exists("market_daily")
return_cmp = read_csv_if_exists("return_comparison")
direction_cmp = read_csv_if_exists("direction_comparison")
paired_preds = read_csv_if_exists("paired_predictions")
period_metrics = read_csv_if_exists("period_metrics")
window_cmp = read_csv_if_exists("window_comparison")
walk_preds = read_csv_if_exists("walk_predictions")
adaptive_metrics = read_csv_if_exists("adaptive_metrics")
adaptive_config = read_json_if_exists("adaptive_config")
latest_forecast = read_csv_if_exists("forecast")
forecast_comparison = read_csv_if_exists("forecast_comparison")
outcome_history = read_csv_if_exists("outcome_history")

## Data inventory: rows, columns, dates, duplicates, missingness

This section measures the files currently on disk. It does not hard-code row counts because they change when the pipeline imports additional price sessions or scores more headlines.

In [ ]:
def date_column(df):
    if df is None: return None
    for c in df.columns:
        if c.lower() in {"date", "session_date", "trading_date", "timestamp", "datetime", "published_at"}:
            return c
    return None

def dataset_summary(name, df):
    if df is None:
        return {"dataset": name, "available": False}
    dcol = date_column(df)
    dates = pd.to_datetime(df[dcol], errors="coerce") if dcol else None
    return {
        "dataset": name, "available": True, "rows": len(df), "columns": df.shape[1],
        "date_column": dcol,
        "date_min": dates.min() if dates is not None else pd.NaT,
        "date_max": dates.max() if dates is not None else pd.NaT,
        "duplicate_rows": int(df.duplicated().sum()),
        "columns_with_missing": int(df.isna().any().sum()),
        "missing_cells": int(df.isna().sum().sum()),
    }

datasets = [
    ("full model dataset", full_df), ("monthly-capped model dataset", capped_df),
    ("scored article corpus", scored_news), ("capped article corpus", capped_news),
    ("full daily sentiment", daily_full), ("capped daily sentiment", daily_capped),
    ("walk-forward predictions", walk_preds), ("adaptive metrics", adaptive_metrics),
]
display(pd.DataFrame([dataset_summary(n, d) for n, d in datasets]))

In [ ]:
for label, df in [("FULL MODEL DATASET", full_df), ("MONTHLY-CAPPED MODEL DATASET", capped_df)]:
    if df is not None:
        print(f"\n{label}: {df.shape}")
        display(pd.DataFrame({"column": df.columns, "dtype": [str(x) for x in df.dtypes],
                              "missing": df.isna().sum().values,
                              "unique": df.nunique(dropna=False).values}))

## Full corpus versus 2026 monthly cap: what is being capped?

The sampling rule keeps the earlier 2021–2025 article history and limits 2026 article volume to at most 500 articles per calendar month. This is **article-level sampling**, followed by rebuilding daily sentiment features and joining them to market sessions. It does not cap the number of market sessions or training rows.

From the latest pipeline log (29 October 2026):
- Scored articles before sampling: **27,819**
- Articles after sampling: **25,759**
- Articles removed: **2,060**
- 2021–2025 articles retained: **21,256**
- 2026 articles before sampling: **6,563**
- Monthly cap: **500**
- 2026 articles after sampling: **4,503** (nine complete/partial months in the log; individual monthly counts below)
- Model dataset: **1,439 market sessions × 18 columns**.

These are a dated snapshot from the recorded run, not a promise that files will always have those counts. The current files loaded above are authoritative for this notebook run.

In [ ]:
if scored_news is not None and capped_news is not None:
    print("Article-level corpus counts from current files")
    display(pd.DataFrame([
        {"version": "Full scored corpus", "rows": len(scored_news)},
        {"version": "After monthly cap", "rows": len(capped_news)},
        {"version": "Removed", "rows": len(scored_news)-len(capped_news)},
    ]))
else:
    print("Article CSVs are not both available. Use the dated pipeline snapshot in the markdown above.")

def monthly_article_counts(df, name):
    if df is None: return None
    dcol = date_column(df)
    if dcol is None:
        print(f"{name}: no obvious date column; columns are {list(df.columns)}")
        return None
    tmp = df.copy()
    tmp[dcol] = pd.to_datetime(tmp[dcol], errors="coerce")
    tmp = tmp.dropna(subset=[dcol])
    tmp["month"] = tmp[dcol].dt.to_period("M").astype(str)
    return tmp.groupby("month").size().rename(name)

monthly_counts = []
for nm, d in [("full_articles", scored_news), ("capped_articles", capped_news)]:
    s = monthly_article_counts(d, nm)
    if s is not None: monthly_counts.append(s)
if monthly_counts:
    month_table = pd.concat(monthly_counts, axis=1).fillna(0).astype(int)
    month_table["articles_removed"] = month_table.get("full_articles", 0) - month_table.get("capped_articles", 0)
    display(month_table.tail(24))
    ax = month_table[[c for c in ["full_articles", "capped_articles"] if c in month_table]].tail(24).plot(
        kind="bar", figsize=(14, 5), title="Article counts by month: full corpus vs sampled corpus")
    ax.set_ylabel("Article rows"); ax.set_xlabel("Month"); plt.xticks(rotation=60); plt.tight_layout(); plt.show()

### Interpretation of the cap

The cap controls the disproportionate influence of months with unusually large numbers of scraped articles (October 2026 was particularly high in the recorded run). It reduces volume-driven imbalance and makes the 2026 sample less dominated by a single month. It also discards information, and the selected subset can add sampling noise. Therefore capped versus uncapped is a sensitivity comparison, not proof that one dataset is intrinsically better.

The cap is **not** a cap on the total dataset. Historical articles from 2021–2025 were retained, and all market sessions remain in the joined model dataset.

## Inspect the market-session datasets

Check date order, duplicates, feature columns, and numeric distributions. The exact column names are read from the local CSVs because the schema may evolve.

In [ ]:
for label, df in [("Full", full_df), ("Monthly-capped", capped_df)]:
    if df is None: continue
    print(f"\n{label} dataset — first 5 rows")
    display(df.head())
    print(f"{label} dataset — last 5 rows")
    display(df.tail())

In [ ]:
def coerce_date_index(df):
    if df is None: return None
    dcol = date_column(df)
    if dcol is None: return df.copy()
    out = df.copy()
    out[dcol] = pd.to_datetime(out[dcol], errors="coerce")
    return out.sort_values(dcol)

for label, df in [("Full", full_df), ("Monthly-capped", capped_df)]:
    if df is None: continue
    tmp = coerce_date_index(df)
    dcol = date_column(tmp)
    print(label, "date field:", dcol)
    if dcol:
        print("monotonic increasing after sort:", tmp[dcol].is_monotonic_increasing)
        print("unique market dates:", tmp[dcol].nunique(), "| rows:", len(tmp))

## Market and news feature exploration

Use the data itself to identify feature names. Histograms and correlations are descriptive only; they do not establish causal influence or out-of-sample predictive value.

In [ ]:
if capped_df is not None:
    numeric_cols = capped_df.select_dtypes(include=np.number).columns.tolist()
    print("Numeric columns in capped dataset:", numeric_cols)
    if numeric_cols:
        display(capped_df[numeric_cols].describe().T)
        capped_df[numeric_cols].hist(figsize=(16, max(5, 2.5*((len(numeric_cols)+3)//4))), bins=35)
        plt.suptitle("Distributions of numeric features — monthly-capped model dataset", y=1.01)
        plt.tight_layout(); plt.show()

In [ ]:
if capped_df is not None:
    numeric = capped_df.select_dtypes(include=np.number)
    if numeric.shape[1] >= 2:
        corr = numeric.corr()
        fig, ax = plt.subplots(figsize=(max(8, 0.75*len(corr.columns)), max(6, 0.65*len(corr.columns))))
        im = ax.imshow(corr, vmin=-1, vmax=1, aspect="auto")
        ax.set_xticks(range(len(corr.columns))); ax.set_xticklabels(corr.columns, rotation=90)
        ax.set_yticks(range(len(corr.columns))); ax.set_yticklabels(corr.columns)
        fig.colorbar(im, ax=ax, label="Pearson correlation")
        ax.set_title("Feature correlation matrix (descriptive)")
        plt.tight_layout(); plt.show()

## Pearson versus Spearman correlation

Pearson correlation measures **linear association**. Spearman correlation is Pearson correlation applied to ranks, so it measures **monotonic association**: as one variable rises, does the other tend to rise or fall, even if the relationship is curved rather than a straight line?

A large absolute Spearman coefficient with a smaller Pearson coefficient can be consistent with a monotonic but nonlinear relationship. It is not a general detector of every nonlinear relationship. For example, a symmetric U-shaped relationship can have both Pearson and Spearman correlations near zero even though the variables are strongly related. Scatterplots and domain reasoning remain important.

Correlation does not establish causation or forecasting utility. For time-series features, autocorrelation, common trends, outliers, and repeated testing can inflate apparent relationships. For predictive claims, estimate correlations within training periods and verify any selected feature with walk-forward out-of-sample performance.

In [ ]:
if capped_df is not None:
    corr_data = capped_df.select_dtypes(include=np.number).replace([np.inf, -np.inf], np.nan)
    # Drop entirely empty or constant columns; pairwise correlations handle remaining missingness.
    corr_data = corr_data.loc[:, corr_data.nunique(dropna=True) > 1]
    pearson_corr = corr_data.corr(method="pearson")
    spearman_corr = corr_data.corr(method="spearman")
    print("Pearson correlation matrix")
    display(pearson_corr.round(3))
    print("Spearman rank-correlation matrix")
    display(spearman_corr.round(3))
    common = pearson_corr.columns.intersection(spearman_corr.columns)
    corr_diff = (spearman_corr.loc[common, common] - pearson_corr.loc[common, common]).abs()
    fig, ax = plt.subplots(figsize=(max(8, .7*len(common)), max(6, .6*len(common))))
    im = ax.imshow(corr_diff, aspect="auto")
    ax.set_xticks(range(len(common))); ax.set_xticklabels(common, rotation=90)
    ax.set_yticks(range(len(common))); ax.set_yticklabels(common)
    fig.colorbar(im, ax=ax, label="|Spearman − Pearson|")
    ax.set_title("Where rank and linear correlations differ most")
    plt.tight_layout(); plt.show()
else:
    print("Capped model dataset unavailable.")

In [ ]:
# Identify pairs where rank correlation is notably stronger than linear correlation.
# This is a screening list, not proof of nonlinear dependence.
if capped_df is not None:
    rows = []
    cols = list(common) if "common" in globals() else []
    for i, a in enumerate(cols):
        for b in cols[i+1:]:
            rp, rs = pearson_corr.loc[a,b], spearman_corr.loc[a,b]
            if pd.notna(rp) and pd.notna(rs):
                rows.append({"feature_1":a, "feature_2":b, "pearson_r":rp,
                             "spearman_rho":rs, "abs_difference":abs(rs-rp),
                             "spearman_stronger_abs":abs(rs)>abs(rp)})
    corr_pairs = pd.DataFrame(rows)
    if len(corr_pairs):
        display(corr_pairs.sort_values("abs_difference", ascending=False).head(25).round(3))

### Scatterplots for the largest Pearson–Spearman disagreements

The next cell draws the top pairs with the largest difference between Pearson and Spearman. Inspect the shape: monotonic curves, saturation, outliers, and U-shaped patterns have different interpretations. If the target/feature columns are in percentage units, axes retain those source units.

In [ ]:
if capped_df is not None and "corr_pairs" in globals() and len(corr_pairs):
    top_pairs = corr_pairs.sort_values("abs_difference", ascending=False).head(6)
    for _, row in top_pairs.iterrows():
        a, b = row["feature_1"], row["feature_2"]
        pair = capped_df[[a,b]].apply(pd.to_numeric, errors="coerce").dropna()
        if len(pair) > 1000:
            pair = pair.sample(1000, random_state=42)
        plt.figure(figsize=(5.5,4))
        plt.scatter(pair[a], pair[b], alpha=.35)
        plt.title(f"{a} vs {b}\nPearson={row['pearson_r']:.2f}, Spearman={row['spearman_rho']:.2f}")
        plt.xlabel(a); plt.ylabel(b); plt.tight_layout(); plt.show()

### Important interpretation limit

Spearman captures monotonic nonlinear patterns, not arbitrary nonlinear dependence. If you suspect a U-shaped or threshold relationship, use scatterplots and consider a separate exploratory method such as mutual information or a spline/GAM model, with feature selection and evaluation kept inside the training folds. Do not search dozens of correlations and report only the strongest ones without accounting for multiple comparisons.

## What the prediction targets mean

The project has two distinct prediction tasks:

1. **Return regression:** predict the next-session return (a continuous target). The baseline predicts the historical mean return available at each training step. Error metrics are MAE and RMSE, usually expressed in percentage points in experiment reports. Lower is better. \(R^2\) below zero means the model performs worse than predicting the evaluation-set mean under that metric definition; it does not by itself tell us whether a strategy is profitable.
2. **Direction classification:** estimate whether the next return is positive. Accuracy is the fraction of correct up/down labels. Brier score is the mean squared error of predicted probabilities against binary outcomes; lower is better. A probability such as 58.6% is not automatically calibrated.

The feature sets compared in the experiment are:
- **Mean baseline** for return.
- **Market + news volume** for return.
- **Market + news + sentiment** for return.
- **Market-only direction** classifier.
- **Sentiment direction** classifier.

The notebook displays saved experiment results rather than silently retraining a different model and calling it the official result.

## Walk-forward validation: why it is used

A random train/test split is inappropriate for this time-ordered forecasting problem because it can train on observations from the future relative to an earlier test point. Walk-forward validation respects chronology:

- fit only on observations available before the prediction date;
- predict the next observation (or next test block, depending on the implementation);
- advance time and repeat;
- compute metrics over the sequence of genuinely out-of-sample predictions.

**Expanding window:** the training set grows as time advances. It retains older regimes and increases the training sample size.

**Rolling window:** training is restricted to the most recent fixed number of sessions (the project compares a 252-session rolling window). It can adapt more quickly to regime changes but discards older data and has fewer training observations.

Neither is universally best. The comparison must be made on the same dates and target definition, using saved predictions and metrics.

In [ ]:
if return_cmp is not None:
    print("Return comparison report")
    display(return_cmp)
if direction_cmp is not None:
    print("Direction comparison report")
    display(direction_cmp)
if window_cmp is not None:
    print("Expanding vs rolling comparison report")
    display(window_cmp)
if period_metrics is not None:
    print("Period-level metrics (useful for regime variation)")
    display(period_metrics)

### Recorded full-period results from the latest sampling experiment

The logged experiment produced **432 out-of-sample predictions** for both original and monthly-sampled datasets, with shared forecast dates from approximately **28 January 2025 to 28 October 2026**. Latest recorded aggregate results:

| Dataset | Return model | MAE (%) | RMSE (%) | R² |
|---|---|---:|---:|---:|
| Full corpus | Mean baseline | 0.6140 | 0.8314 | -0.0035 |
| Full corpus | Market + news volume | 0.6247 | 0.8383 | -0.0201 |
| Full corpus | Market + news + sentiment | 0.6315 | 0.8431 | -0.0318 |
| Monthly sampled | Mean baseline | 0.6140 | 0.8314 | -0.0035 |
| Monthly sampled | Market + news volume | 0.6215 | 0.8358 | -0.0139 |
| Monthly sampled | Market + news + sentiment | 0.6285 | 0.8409 | -0.0265 |

Interpretation: the mean-return baseline had the lowest MAE and RMSE among these return regressors. The capped corpus slightly improved the two feature models relative to the full corpus, but did not make them outperform the baseline. The negative \(R^2\) values also do not support a strong predictive claim.

The logged direction results were modest: sentiment direction accuracy was about **0.5046** for both datasets in that run; market-only accuracy was about **0.4676** (full) and **0.4630** (sampled). The sentiment classifier’s Brier score was around **0.258**, so probability quality is not established. Use the currently loaded reports above if they differ from this dated snapshot.

## Compare expanding versus rolling windows carefully

The selector evaluates recent genuine out-of-sample predictions over a recent two-month window and compares expanding versus rolling 252-session training for the candidate model families. A rolling window is not automatically better simply because it is more recent. Look for:

- lower return MAE/RMSE on the same prediction dates;
- direction performance that is better than a naive reference and is not an artefact of a few observations;
- enough observations in each period;
- stable results across periods rather than one isolated good month;
- no leakage from future prices or news unavailable at the 15:45 IST cutoff.

A recent window of roughly 41 predictions is useful for adaptation but statistically noisy. It should not be interpreted as a definitive ranking of strategies.

In [ ]:
if walk_preds is not None:
    print("Walk-forward prediction columns:", list(walk_preds.columns))
    display(walk_preds.head(10))
    print("Rows:", len(walk_preds))
if window_cmp is not None:
    # Show the full table, plus a convenient sorted view where metric columns exist.
    metric_candidates = [c for c in ["MAE_pct", "RMSE_pct", "accuracy", "brier_score"] if c in window_cmp.columns]
    if metric_candidates:
        display(window_cmp.sort_values(metric_candidates[0], ascending=(metric_candidates[0] != "accuracy")))

## Walk-forward visual dashboard

These plots use the experiment CSVs already produced by the pipeline. They compare expanding and rolling validation where the saved output contains the relevant fields. They do not retrain models or manufacture missing metrics. Some column names differ between experiment versions, so the cells inspect columns and choose known aliases.

In [ ]:
def find_col(df, aliases):
    if df is None: return None
    lower = {str(c).lower(): c for c in df.columns}
    for a in aliases:
        if a.lower() in lower: return lower[a.lower()]
    for c in df.columns:
        if any(a.lower() in str(c).lower() for a in aliases):
            return c
    return None

if window_cmp is not None:
    print("Available walk-forward metric columns:", list(window_cmp.columns))
    display(window_cmp.head())

In [ ]:
if window_cmp is not None:
    mode_col = find_col(window_cmp, ["window_mode", "window", "mode"])
    model_col = find_col(window_cmp, ["model", "model_name", "candidate"])
    mae_col = find_col(window_cmp, ["mae_pct", "mae"])
    rmse_col = find_col(window_cmp, ["rmse_pct", "rmse"])
    acc_col = find_col(window_cmp, ["accuracy", "direction_accuracy"])
    brier_col = find_col(window_cmp, ["brier_score", "brier"])
    ds_col = find_col(window_cmp, ["dataset"])
    if mode_col and model_col:
        plot_df = window_cmp.copy()
        plot_df["comparison_label"] = plot_df[mode_col].astype(str) + " | " + plot_df[model_col].astype(str)
        for metric, title, lower_better in [
            (mae_col, "Walk-forward comparison: return MAE (lower is better)", True),
            (rmse_col, "Walk-forward comparison: return RMSE (lower is better)", True),
            (acc_col, "Walk-forward comparison: direction accuracy (higher is better)", False),
            (brier_col, "Walk-forward comparison: direction Brier score (lower is better)", True),
        ]:
            if metric:
                plot_df[metric] = pd.to_numeric(plot_df[metric], errors="coerce")
                view = plot_df.dropna(subset=[metric])
                if ds_col and view[ds_col].nunique() > 1:
                    for dataset_name, g in view.groupby(ds_col):
                        g = g.sort_values(metric, ascending=lower_better)
                        plt.figure(figsize=(11, max(4, .38*len(g))))
                        plt.barh(g["comparison_label"], g[metric])
                        plt.title(f"{title}\nDataset: {dataset_name}")
                        plt.xlabel(metric); plt.tight_layout(); plt.show()
                elif len(view):
                    view = view.sort_values(metric, ascending=lower_better)
                    plt.figure(figsize=(11, max(4, .38*len(view))))
                    plt.barh(view["comparison_label"], view[metric])
                    plt.title(title); plt.xlabel(metric); plt.tight_layout(); plt.show()
    else:
        print("Could not identify both a window-mode and model column. Inspect the table above and adapt aliases.")
else:
    print("No walk_forward_window_comparison.csv found; run the existing experiment script if you want to refresh historical comparison outputs.")

### Rolling-period performance and regime variation

Aggregate metrics can conceal regime dependence. The next plots show how saved period-level metrics vary over time. A short rolling-window period can look unusually good or bad because it contains few test predictions, so always read prediction counts and date ranges alongside the metric.

In [ ]:
if period_metrics is not None:
    print("Period metric columns:", list(period_metrics.columns))
    display(period_metrics.head(12))
    period_col = find_col(period_metrics, ["period", "period_label", "quarter", "window_period", "start_date"])
    model_col_p = find_col(period_metrics, ["model", "model_name"])
    mae_col_p = find_col(period_metrics, ["mae_pct", "mae"])
    rmse_col_p = find_col(period_metrics, ["rmse_pct", "rmse"])
    count_col_p = find_col(period_metrics, ["n_predictions", "n_test", "count"])
    if period_col and (mae_col_p or rmse_col_p):
        plot = period_metrics.copy()
        plot[period_col] = plot[period_col].astype(str)
        for col, title in [(mae_col_p, "Return MAE by evaluation period"),
                           (rmse_col_p, "Return RMSE by evaluation period")]:
            if col:
                plot[col] = pd.to_numeric(plot[col], errors="coerce")
                fig, ax = plt.subplots(figsize=(13, 5))
                if model_col_p:
                    for name, g in plot.groupby(model_col_p):
                        ax.plot(g[period_col], g[col], marker="o", label=str(name))
                    ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
                else:
                    ax.plot(plot[period_col], plot[col], marker="o")
                ax.set_title(title); ax.set_ylabel(col); ax.set_xlabel("Period")
                ax.tick_params(axis="x", rotation=60); ax.grid(alpha=.25)
                plt.tight_layout(); plt.show()
    else:
        print("Could not identify period and metric columns for line plots; table above remains available.")

### Actual versus predicted returns and residual diagnostics

When the saved walk-forward prediction table contains actual and predicted return columns, these graphs help identify bias, missed volatility, and whether a model's apparent average performance is driven by a few large errors.

In [ ]:
if walk_preds is not None:
    print("Prediction columns:", list(walk_preds.columns))
    date_c = find_col(walk_preds, ["date", "forecast_date", "target_date", "session_date"])
    actual_c = find_col(walk_preds, ["actual_return", "y_true", "actual", "true_return"])
    pred_c = find_col(walk_preds, ["predicted_return", "prediction", "y_pred", "predicted", "forecast_return"])
    model_c = find_col(walk_preds, ["model", "model_name", "candidate"])
    if actual_c and pred_c:
        plot = walk_preds.copy()
        plot[actual_c] = pd.to_numeric(plot[actual_c], errors="coerce")
        plot[pred_c] = pd.to_numeric(plot[pred_c], errors="coerce")
        plot = plot.dropna(subset=[actual_c, pred_c])
        if model_c:
            names = list(plot[model_c].dropna().unique())
            for name in names[:8]:
                g = plot[plot[model_c] == name]
                plt.figure(figsize=(12, 4))
                if date_c:
                    x = pd.to_datetime(g[date_c], errors="coerce")
                    plt.plot(x, g[actual_c], label="Actual", alpha=.7)
                    plt.plot(x, g[pred_c], label="Predicted", alpha=.7)
                    plt.xticks(rotation=45)
                else:
                    plt.plot(g[actual_c].to_numpy(), label="Actual", alpha=.7)
                    plt.plot(g[pred_c].to_numpy(), label="Predicted", alpha=.7)
                plt.title(f"Walk-forward actual vs predicted — {name}")
                plt.legend(); plt.tight_layout(); plt.show()
                residual = g[actual_c] - g[pred_c]
                plt.figure(figsize=(6, 4))
                plt.scatter(g[pred_c], residual, alpha=.45)
                plt.axhline(0, linestyle="--")
                plt.title(f"Residuals vs prediction — {name}")
                plt.xlabel("Predicted return"); plt.ylabel("Actual − predicted")
                plt.tight_layout(); plt.show()
        else:
            plt.figure(figsize=(12, 4))
            if date_c:
                plt.plot(pd.to_datetime(plot[date_c], errors="coerce"), plot[actual_c], label="Actual")
                plt.plot(pd.to_datetime(plot[date_c], errors="coerce"), plot[pred_c], label="Predicted")
            else:
                plt.plot(plot[actual_c].to_numpy(), label="Actual")
                plt.plot(plot[pred_c].to_numpy(), label="Predicted")
            plt.title("Walk-forward actual vs predicted"); plt.legend(); plt.tight_layout(); plt.show()
    else:
        print("Actual/predicted columns not recognized. Use the column listing above to map the names explicitly.")

## PCA: what it does, what it can tell us, and why it was not adopted

**Principal Component Analysis (PCA)** transforms correlated numeric predictors into orthogonal linear combinations called principal components. Component 1 captures the largest possible variance; subsequent components capture the largest remaining variance subject to orthogonality. PCA is unsupervised: it does not know which direction or return is the target.

### Correct preprocessing for a time-series model
- Exclude target columns and identifiers/dates from PCA.
- Standardize predictors using statistics fitted on the training fold only.
- Fit PCA on the training fold only.
- Transform the test fold using the already-fitted scaler and PCA.
- Fit the downstream predictor on training-fold component scores.
- Evaluate predictions on chronologically later data.

Fitting PCA once on the entire dataset before walk-forward validation leaks information about the future predictor distribution, even though PCA does not use the target. For a fair comparison, PCA must be nested inside each walk-forward fold.

### Why PCA was not used in the official forecast
PCA can reduce collinearity and dimensionality, but maximum predictor variance is not the same as maximum predictive signal. Components can discard low-variance features that contain useful information, and the transformed components are less directly interpretable than market/news variables. Most importantly, the saved walk-forward results available for this project did not demonstrate a reliable improvement over the mean-return baseline. Therefore PCA remains an exploratory experiment, not part of the official adaptive forecast, unless a leakage-safe walk-forward comparison demonstrates consistent improvement.

The notebook below runs a **diagnostic PCA** on numeric predictors from the current capped dataset. It is explicitly descriptive and is **not** a valid out-of-sample performance result. The later nested walk-forward PCA example is the appropriate template for model evaluation.

In [ ]:
try:
    from sklearn.preprocessing import StandardScaler
    from sklearn.decomposition import PCA
    SKLEARN_AVAILABLE = True
except Exception as e:
    SKLEARN_AVAILABLE = False
    print("scikit-learn unavailable:", e)

if SKLEARN_AVAILABLE and capped_df is not None:
    # Exclude obvious date/id/target-like fields; review printed list before interpreting.
    excluded_tokens = ("date", "time", "target", "label", "future", "next_return", "next_return_pct", "direction")
    candidate_cols = []
    for c in capped_df.select_dtypes(include=np.number).columns:
        if not any(tok in c.lower() for tok in excluded_tokens):
            candidate_cols.append(c)
    print("PCA numeric predictor candidates:", candidate_cols)
    if len(candidate_cols) >= 2:
        X = capped_df[candidate_cols].replace([np.inf, -np.inf], np.nan)
        X = X.fillna(X.median(numeric_only=True))
        Xs = StandardScaler().fit_transform(X)
        pca = PCA().fit(Xs)
        cum = np.cumsum(pca.explained_variance_ratio_)
        pca_summary = pd.DataFrame({
            "component": [f"PC{i+1}" for i in range(len(pca.explained_variance_ratio_))],
            "explained_variance_ratio": pca.explained_variance_ratio_,
            "cumulative_variance": cum
        })
        display(pca_summary)
        plt.figure(figsize=(10, 4))
        plt.plot(range(1, len(cum)+1), cum, marker="o")
        plt.axhline(.80, linestyle="--", label="80% variance")
        plt.axhline(.90, linestyle=":", label="90% variance")
        plt.xlabel("Number of components"); plt.ylabel("Cumulative explained variance")
        plt.title("Diagnostic PCA — full capped dataset (descriptive only)")
        plt.legend(); plt.grid(alpha=.25); plt.tight_layout(); plt.show()
        loadings = pd.DataFrame(pca.components_.T, index=candidate_cols,
                                columns=[f"PC{i+1}" for i in range(len(candidate_cols))])
        print("Loadings for first 3 PCs")
        display(loadings.iloc[:, :min(3, loadings.shape[1])])
    else:
        print("Not enough numeric predictors for PCA.")
else:
    print("PCA skipped because scikit-learn or capped dataset is unavailable.")

### PCA results: how to report them without overclaiming

This cell computes the current file's variance-explained curve and loadings. Record:
1. number of predictors included and excluded;
2. number of components needed for 80% and 90% variance;
3. whether the first components are dominated by redundant market variables or news aggregates;
4. most importantly, whether a **nested walk-forward** PCA model improves MAE/RMSE versus the same model without PCA and versus the mean baseline.

Do not conclude “PCA worked” from a high cumulative explained-variance percentage. That percentage describes \(X\), not prediction accuracy.

## A leakage-safe PCA walk-forward template

This optional cell is a **template**, not a reconstruction of the official model. It only runs if target and date columns can be identified. It fits the scaler and PCA inside each expanding fold. Review the printed target/date/feature choices before relying on the result. The official pipeline's own training code remains authoritative for the deployed forecast.

In [ ]:
# Conservative column detection: print candidates and do not guess silently.
if capped_df is not None and SKLEARN_AVAILABLE:
    date_candidates = [c for c in capped_df.columns if c.lower() in {"date","session_date","trading_date"}]
    target_candidates = [c for c in capped_df.columns if c.lower() in {
        "target_return", "next_return", "next_return_pct", "target_return_pct", "forward_return", "y_return"
    }]
    print("Date candidates:", date_candidates)
    print("Return-target candidates:", target_candidates)
    print("All columns:", list(capped_df.columns))
    print("If the project uses different names, adapt the explicit DATE_COL and TARGET_COL below before running.")

In [ ]:
# Set these only after checking the previous cell's output.
DATE_COL = next((c for c in (date_candidates if capped_df is not None and SKLEARN_AVAILABLE else []) if c in capped_df.columns), None)
TARGET_COL = next((c for c in (target_candidates if capped_df is not None and SKLEARN_AVAILABLE else []) if c in capped_df.columns), None)

if capped_df is None or not SKLEARN_AVAILABLE or DATE_COL is None or TARGET_COL is None:
    print("Template not run: date/return target could not be identified safely. No assumptions made.")
else:
    from sklearn.linear_model import Ridge
    from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
    temp = capped_df.copy()
    temp[DATE_COL] = pd.to_datetime(temp[DATE_COL], errors="coerce")
    temp = temp.dropna(subset=[DATE_COL, TARGET_COL]).sort_values(DATE_COL).reset_index(drop=True)
    drop_cols = {DATE_COL, TARGET_COL}
    feature_cols = [c for c in temp.select_dtypes(include=np.number).columns if c not in drop_cols]
    Xraw = temp[feature_cols].replace([np.inf, -np.inf], np.nan)
    y = pd.to_numeric(temp[TARGET_COL], errors="coerce")
    valid = y.notna()
    temp, Xraw, y = temp.loc[valid].reset_index(drop=True), Xraw.loc[valid].reset_index(drop=True), y.loc[valid].reset_index(drop=True)
    min_train = min(500, max(100, int(len(temp)*0.5)))
    records = []
    for i in range(min_train, len(temp)):
        X_train_raw, X_test_raw = Xraw.iloc[:i], Xraw.iloc[[i]]
        y_train, y_test = y.iloc[:i], y.iloc[[i]]
        med = X_train_raw.median()
        X_train_raw, X_test_raw = X_train_raw.fillna(med), X_test_raw.fillna(med)
        scaler = StandardScaler().fit(X_train_raw)
        X_train_s, X_test_s = scaler.transform(X_train_raw), scaler.transform(X_test_raw)
        pca_fold = PCA(n_components=0.90, svd_solver="full").fit(X_train_s)
        X_train_p, X_test_p = pca_fold.transform(X_train_s), pca_fold.transform(X_test_s)
        model = Ridge(alpha=1.0).fit(X_train_p, y_train)
        pred = float(model.predict(X_test_p)[0])
        records.append({"date": temp.loc[i, DATE_COL], "actual": float(y_test.iloc[0]),
                        "predicted": pred, "components": pca_fold.n_components_})
    pca_oos = pd.DataFrame(records)
    print("Nested expanding-window PCA predictions:", len(pca_oos))
    if len(pca_oos):
        mae = mean_absolute_error(pca_oos.actual, pca_oos.predicted)
        rmse = mean_squared_error(pca_oos.actual, pca_oos.predicted)**0.5
        baseline_preds = []
        for i in range(min_train, len(temp)):
            baseline_preds.append(float(y.iloc[:i].mean()))
        base_mae = mean_absolute_error(pca_oos.actual, baseline_preds)
        base_rmse = mean_squared_error(pca_oos.actual, baseline_preds)**0.5
        display(pd.DataFrame([
            {"model":"Nested PCA + Ridge", "MAE":mae, "RMSE":rmse},
            {"model":"Expanding mean baseline", "MAE":base_mae, "RMSE":base_rmse},
        ]))
        display(pca_oos.tail())
        ax = pca_oos.set_index("date")[["actual","predicted"]].plot(figsize=(14,4), alpha=.8)
        ax.set_title("Nested expanding-window PCA regression: actual vs predicted")
        ax.set_ylabel(TARGET_COL); plt.tight_layout(); plt.show()

## Adaptive model selection: current production logic

The regular pipeline now runs adaptive selection as part of generating the official forecast. It evaluates recent out-of-sample predictions (roughly the latest two months), compares expanding and rolling-window candidates, and stores:
- `adaptive_model_selection.csv` — recent candidate metrics;
- `adaptive_model_selection.json` — selected configuration;
- `latest_forecast.csv` — the official forecast using the selected return/direction configurations.

The return selector includes the mean-return baseline and feature models. The selection rule requires a feature model to beat the expanding mean-return baseline by the configured **2% MAE margin**; otherwise the baseline is selected. The direction selector is separate because direction classification and return regression have different targets and metrics.

In [ ]:
print("Adaptive configuration")
if adaptive_config is not None:
    display(pd.json_normalize(adaptive_config).T.rename(columns={0:"value"}))
else:
    print("No adaptive_model_selection.json found.")
print("\nAdaptive metrics")
if adaptive_metrics is not None:
    display(adaptive_metrics)
else:
    print("No adaptive_model_selection.csv found.")

### Latest recorded run (29 October 2026)

The pipeline reported:
- **41 recent out-of-sample predictions**, evaluation period **28 August–28 October 2026**.
- Selected return model: **baseline / expanding**.
- Selected direction model: **sentiment / expanding**.
- Selection note: no feature model beat the expanding mean-return baseline by the required 2% MAE margin.
- Adaptive return forecast: **+0.0379%**.
- Direction model probability of UP: **58.6%**.

The model's independent market-only and market+sentiment forecasts were also printed by the pipeline, but the adaptive selected return forecast is the one tied to the selection configuration. A 58.6% model probability should be treated as an uncalibrated model output, not as a literal 58.6% real-world chance unless calibration has been demonstrated.

## Latest official forecast and prospective evaluation

Forecast generation and forecast evaluation are different things. The next-session forecast is made before the target return is known; outcome tracking later joins the saved forecast to the actual market session. Pending forecasts must not be counted as evaluated predictions.

In [ ]:
for key, df in [("latest forecast", latest_forecast), ("side-by-side forecast", forecast_comparison),
                ("prospective outcome history", outcome_history)]:
    print("\n", key.upper())
    if df is None:
        print("Not available.")
    else:
        print("Shape:", df.shape)
        display(df.tail(10))

In the latest recorded run, the price refresh failed to obtain new rows from Yahoo Finance, but the pipeline retained existing prices through **29 October 2026** and continued successfully. Outcome tracking reported **0 newly evaluated forecasts**, **3 pending forecasts**, and **3 legacy/other model forecasts already evaluated**; none were official capped-model evaluated forecasts yet. That is expected when no later target session is available, not a pipeline failure.

The small prospective sample is not enough to infer live performance. The long historical walk-forward evaluation and prospective outcome history answer different questions and should remain separate.

## News cutoff, leakage, and timing

The pipeline uses a **15:45 IST news cutoff**. For a forecast issued for the next session, a same-date news feature is valid only if the article was published and available by that cutoff. If news published after the cutoff is aggregated into the same day's features, it leaks information that would not have been available at forecast time.

For any future validation or revised model:
- use the publication timestamp, not just calendar date;
- apply the same cutoff rule in training and live inference;
- keep feature construction identical between historical evaluation and forecast generation;
- avoid using a full-day sentiment aggregate when it includes after-cutoff headlines.

## Why the final model uses the adaptive setup instead of PCA or a fixed feature model

The final choice is conservative and evidence-driven:

1. **Walk-forward performance is the deciding evidence.** In the recorded 432-prediction comparison, the mean-return baseline had lower MAE/RMSE than both feature models. Adding news volume and sentiment did not improve return regression sufficiently.
2. **Capping improves data-volume control, not necessarily predictive skill.** The monthly cap reduced 2026 article volume and slightly improved feature-model errors versus full corpus, but feature models still did not beat the baseline.
3. **PCA is not automatically predictive.** It maximizes explained variance in the predictors, not association with the future return. It also makes the feature representation less interpretable. PCA would need a nested walk-forward test and a clear improvement over baseline before becoming part of the official model.
4. **Expanding and rolling windows address different regimes.** Expanding retains long history; rolling emphasizes recent observations. The selector compares both based on recent OOS results instead of assuming one always wins.
5. **The baseline fallback prevents unjustified complexity.** The 2% MAE margin is a guardrail: if feature models do not outperform the baseline by enough, the return forecast uses the baseline.
6. **Direction is selected independently.** The current recent window selected sentiment/expanding for direction even though the return selector chose baseline/expanding. This is possible because direction and regression are distinct tasks; it does not mean the direction probabilities are calibrated.
7. **Prospective validation is still pending.** The latest run has not yet accumulated official capped-model forecast outcomes. No trading profitability claim is supported.

This is a model-selection policy, not a guarantee that the selected candidate will remain best. Keep the selector's decisions and realized outcomes recorded over time.

## Reproducibility and audit checklist

Before using the notebook output in a report:
- [ ] Record the date/time of the pipeline run and the exact data files analysed.
- [ ] Verify full/capped article counts and monthly counts from the current CSVs.
- [ ] Confirm that the market-session count and 18-column dataset shape match the pipeline log.
- [ ] Inspect the saved walk-forward metrics and confirm the evaluation dates and prediction counts.
- [ ] Compare expanding and rolling windows on the same out-of-sample dates.
- [ ] Keep PCA out of the official model unless its nested walk-forward results are reproducibly better.
- [ ] Confirm the adaptive JSON selection agrees with the printed forecast.
- [ ] Distinguish forecasts still pending from outcomes already evaluated.
- [ ] Verify every news feature respects the 15:45 IST cutoff.
- [ ] Do not claim profitability, statistical significance, or calibrated probabilities without additional evidence.

### One-sentence summary

The project now uses a monthly-capped 2026 news corpus to control article-volume imbalance, evaluates candidate models with chronological walk-forward validation, and selects return and direction models separately using recent out-of-sample performance; because feature models and exploratory PCA have not demonstrated a reliable gain over the mean-return baseline, the official return selector falls back to baseline/expanding while the direction selector currently uses sentiment/expanding.